In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/model.safetensors.index.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/model-00003-of-00003.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/README.md
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/tokenizer.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/model-00001-of-00003.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/tokenizer_config.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/model-00002-of-00003.safetensors
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/special_tokens_map.json
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/.gitattributes
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/tokenizer.model
/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2/generation_con

In [2]:
import pandas as pd
import glob; d = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0].rsplit("/", 1)[0] + "/"; print(d)
train = pd.read_csv(d + "train.csv")
print(train.shape)
print(train[["winner_model_a", "winner_model_b", "winner_tie"]].mean())
train.head()

/kaggle/input/competitions/llm-classification-finetuning/
(57477, 9)
winner_model_a    0.349079
winner_model_b    0.341911
winner_tie        0.309011
dtype: float64


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [3]:
# Baseline: word counts (TF-IDF) + logistic regression
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

test = pd.read_csv(d + "test.csv")
cols = ["prompt", "response_a", "response_b"]

# Turn each text column into numbers
vecs = {c: TfidfVectorizer(max_features=20000).fit(train[c]) for c in cols}
feats = lambda df: hstack([vecs[c].transform(df[c]) for c in cols])

# Label: 0 = A wins, 1 = B wins, 2 = tie
y = train[["winner_model_a", "winner_model_b", "winner_tie"]].values.argmax(1)
model = LogisticRegression(max_iter=1000).fit(feats(train), y)

# Predict probabilities for the test set and save them
sub = pd.DataFrame(model.predict_proba(feats(test)), columns=["winner_model_a", "winner_model_b", "winner_tie"])
sub.insert(0, "id", test["id"])
sub.to_csv("submission.csv", index=False)
sub

,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.099712,0.284002,0.616286
1,211333,0.511547,0.386726,0.101727
2,1233961,0.400965,0.448025,0.151011


In [4]:
# Steps 1+2: hold out 20% of rows to measure, add comparison features, use LightGBM
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

# Numbers describing one text column: length, words, lines, code blocks, list items, refusals
stats = lambda s: pd.DataFrame({"len": s.str.len(), "words": s.str.split().str.len(), "lines": s.str.count(r"\\n"), "code": s.str.count(chr(96) * 3), "lists": s.str.count(r"\\n\s*[-*\d]"), "sorry": s.str.contains("sorry|cannot|unable", case=False).astype(int)})

# Features for the prompt, for A, for B, and the A-minus-B differences
feats2 = lambda df: pd.concat([stats(df["prompt"]).add_prefix("p_"), stats(df["response_a"]).add_prefix("a_"), stats(df["response_b"]).add_prefix("b_"), (stats(df["response_a"]) - stats(df["response_b"])).add_prefix("diff_")], axis=1)
X = feats2(train)

# Step 1: split rows into 80% for training and 20% for checking
tr, va = train_test_split(np.arange(len(train)), test_size=0.2, random_state=0, stratify=y)

# Score of the old baseline on the held-out 20%
F = feats(train).tocsr()
base = LogisticRegression(max_iter=1000).fit(F[tr], y[tr])
print("baseline validation log loss:", log_loss(y[va], base.predict_proba(F[va])))

# Step 2: LightGBM on the new features, scored on the same 20%
params = dict(n_estimators=400, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
gbm = lgb.LGBMClassifier(**params).fit(X.iloc[tr], y[tr])
print("new model validation log loss:", log_loss(y[va], gbm.predict_proba(X.iloc[va])))

# Retrain on all rows and overwrite submission.csv with the new model's predictions
gbm = lgb.LGBMClassifier(**params).fit(X, y)
sub = pd.DataFrame(gbm.predict_proba(feats2(test)), columns=["winner_model_a", "winner_model_b", "winner_tie"])
sub.insert(0, "id", test["id"])
sub.to_csv("submission.csv", index=False)
sub

baseline validation log loss: 1.1265533531021614
new model validation log loss: 1.038992914827527


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.250327,0.259735,0.489938
1,211333,0.489771,0.224247,0.285982
2,1233961,0.252192,0.421170,0.326637


In [5]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Step 5: Gemma + LightGBM blend, using only the first GPU
!pip uninstall -y -q torchao
import json, torch
from scipy.special import softmax
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import PeftModel

# Find the attached Gemma folder and the folder with the saved LoRA weights
gdir = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "gemma-2-2b" in p][0].rsplit("/", 1)[0]
adir = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)[0].rsplit("/", 1)[0]
print(gdir, adir)

# Load Gemma in half precision and attach the trained LoRA weights
tok = AutoTokenizer.from_pretrained(gdir)
base = AutoModelForSequenceClassification.from_pretrained(gdir, num_labels=3, dtype=torch.float16, device_map="cuda:0")
base.config.pad_token_id = tok.pad_token_id
net = PeftModel.from_pretrained(base, adir)

# Build the same text format that was used in training, then turn it into tokens
clean = lambda s, n: " ".join(str(x) for x in json.loads(s)).encode("utf-8", "ignore").decode()[:n]
text = lambda df: ["Prompt: " + clean(p, 500) + "\n\nResponse A: " + clean(a, 1200) + "\n\nResponse B: " + clean(b, 1200) + "\n\nWhich response is better?" for p, a, b in zip(df["prompt"], df["response_a"], df["response_b"])]
enc = lambda df: Dataset.from_dict({"text": text(df)}).map(lambda b: tok(b["text"], truncation=True, max_length=768), batched=True, remove_columns=["text"])
args = TrainingArguments(output_dir="/kaggle/temp/out", per_device_eval_batch_size=8, fp16=True, report_to="none")
trainer = Trainer(model=net, args=args, data_collator=DataCollatorWithPadding(tok))
predict = lambda df: softmax(trainer.predict(enc(df)).predictions.astype("float32"), axis=1)

# Pick the blend weight on 2,000 held-out rows that Gemma never trained on
chk = train.sample(2000, random_state=1).index.values  # the same 2,000 battles that the Gemma training notebook kept out
rest = np.setdiff1d(np.arange(len(train)), chk)  # every other row, used to fit LightGBM for this check
p_gem = predict(train.iloc[chk])
p_gbm = lgb.LGBMClassifier(**params).fit(X.iloc[rest], y[rest]).predict_proba(X.iloc[chk])
scores = {w / 10: log_loss(y[chk], w / 10 * p_gem + (1 - w / 10) * p_gbm) for w in range(11)}
print(scores)
best = min(scores, key=scores.get)
print("best Gemma weight:", best, "-> blended log loss:", scores[best])

# Predict the test set with both models, mix with the best weight, and write submission.csv
sub = pd.DataFrame(best * predict(test) + (1 - best) * gbm.predict_proba(feats2(test)), columns=["winner_model_a", "winner_model_b", "winner_tie"])
sub.insert(0, "id", test["id"])
sub.to_csv("submission.csv", index=False)
sub

/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2 /kaggle/input/notebooks/phankhnhng/notebook1087d64cfa/gemma2b-lora


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Gemma2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b/2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: T

{0.0: 1.0348371571692527, 0.1: 1.02560040543773, 0.2: 1.0177992321697786, 0.3: 1.0113082444363681, 0.4: 1.0060499339901163, 0.5: 1.0019830164967218, 0.6: 0.9990983595370481, 0.7: 0.9974208013825325, 0.8: 0.9970181314533532, 0.9: 0.9980282365063577, 1.0: 1.0007595935267366}
best Gemma weight: 0.8 -> blended log loss: 0.9970181314533532


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.226993,0.355271,0.417736
1,211333,0.267676,0.495515,0.236809
2,1233961,0.159695,0.528764,0.311541


##### 